<a href="https://colab.research.google.com/github/ThiagoDamasz/capacitacao-IA-CS-I/blob/main/ATV1_Capacitacao_CS%26I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importando as bibliotecas necessárias

In [1]:
!pip install scikit-fuzzy #instalando biblioteca para lógica fuzzy
import numpy as np
import skfuzzy as fuzz
import random
from skfuzzy import control as ctrl
from sklearn.datasets import load_iris #importando o dataset das flores
from sklearn.model_selection import train_test_split #divide os dados em dados de treino e teste
from sklearn.tree import DecisionTreeClassifier #algoritimo da árvore de Decisão
from sklearn.metrics import accuracy_score #cálcula a acurácia do modelo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 10.0 MB/s eta 0:00:00


# Carregando e separando os dados



Primeiro é necessário separar os dados. Os dados em "x" são as features (características) das flores. Já os dados em "y" são as labels (rótulos), ou seja a espécie de cada flor.



A função **train_test_split** divide os dados em treino e teste.
O parâmetro **test_size** define qual porcentagem dos dados será utilizada para
o teste. Já o **random_state** controla a aleatoriedade da divisão dos dados, garantindo que seja igual na maioria das vezes.




In [2]:
dados = load_iris() #carregando os dados
x = dados.data #armazenando os dados em x
y = dados.target #armazenando os dados em y

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42) # dividindo o dataset em treino e teste


# Árvore de Decisão

Uma **Árvore de Decisão** é um algoritmo de Machine Learning supervisionado usado para classificação ou regressão. A ideia dele é tomar decisões fazendo perguntas sequenciais sobre os dados, formando uma estrutura parecida com uma árvore.

In [ ]:
arvore_decisao = DecisionTreeClassifier(random_state=42) #criando o modelo da árvore de decisão
arvore_decisao.fit(x_train, y_train) #executando o treinamento da árvore

DecisionTreeClassifier(random_state=42)

O método "predict()" pede ao modelo para prever a classe de novos dados.
Ele pega as características de entrada e retorna a classe prevista.

In [ ]:
y_pred = arvore_decisao.predict(x_test) # realizando a predição
acuracia = accuracy_score(y_test, y_pred) # cálcula a acurácia do modelo.

# Resultados

In [ ]:
print("=== ML (Decision Tree) ===")
print("Acurácia:", round(acuracia, 4))
print("Exemplo de previsões:", y_pred[:10])

=== ML (Decision Tree) ===
Acurácia: 1.0
Exemplo de previsões: [1 0 2 1 1 0 1 2 1 1]


# Lógica fuzzy


Nesta célula é implementado um sistema de lógica fuzzy utilizando o Iris Dataset e a biblioteca scikit-fuzzy.

Primeiramente, é utilizada a variável petal_length (comprimento da pétala) como entrada do sistema. Em seguida são definidas duas variáveis fuzzy:

comprimento (ctrl.Antecedent) → variável de entrada do sistema.

classe (ctrl.Consequent) → variável de saída, representando a espécie da flor.

Depois são criadas funções de pertinência triangulares (fuzz.trimf) para definir os conjuntos fuzzy de entrada (pequeno, medio, grande) e de saída (setosa, versicolor, virginica).

Com esses conjuntos são definidas regras fuzzy (ctrl.Rule) que relacionam o comprimento da pétala com a espécie da flor. Por fim, o sistema fuzzy é criado (ControlSystem) e executado (ControlSystemSimulation) para classificar uma flor com comprimento = 5.0, retornando a classe prevista.

In [ ]:
petal_length =x[:,2]

comprimento = ctrl.Antecedent(np.arange(0, 8, 0.1), 'comprimento') #Variável fuzzy de entrada
classe = ctrl.Consequent(np.arange(0, 3, 1), 'classe')#Variável fuzzy de saída

comprimento['pequeno'] = fuzz.trimf(comprimento.universe, [0,0,3])
comprimento['medio'] = fuzz.trimf(comprimento.universe, [2,4,6])
comprimento['grande'] = fuzz.trimf(comprimento.universe, [5,7,7])

classe['setosa'] = fuzz.trimf(classe.universe, [0,0,1])
classe['versicolor'] = fuzz.trimf(classe.universe, [0,1,2])
classe['virginica'] = fuzz.trimf(classe.universe, [1,2,2])

regra1 = ctrl.Rule(comprimento['pequeno'], classe['setosa'])
regra2 = ctrl.Rule(comprimento['medio'], classe['versicolor'])
regra3 = ctrl.Rule(comprimento['grande'], classe['virginica'])

sistema = ctrl.ControlSystem([regra1, regra2, regra3])
simulador = ctrl.ControlSystemSimulation(sistema)

simulador.input['comprimento'] = 5.0
simulador.compute()

print("Resultado fuzzy:", simulador.output['classe'])

Resultado fuzzy: 1.0


# Algoritmo genético

Nesta etapa foi utilizado um Genetic Algorithm para encontrar automaticamente um valor de threshold para o comprimento da pétala (petal_length) no Iris Dataset. O código começa utilizando apenas a feature petal_length e define a função fitness(), que avalia a qualidade de cada solução calculando a acurácia da classificação em relação às classes reais do dataset.

Em seguida, o algoritmo cria uma população inicial de valores aleatórios que representam possíveis thresholds. Ao longo de várias gerações, cada solução é avaliada, as melhores são selecionadas e novas soluções são geradas através do cruzamento entre indivíduos e da aplicação de pequenas mutações aleatórias. Ao final do processo evolutivo, o algoritmo retorna o melhor valor de threshold encontrado e a acurácia correspondente para a classificação das flores.

In [4]:
# usar apenas petal length
petal_length = x[:,2]

# função fitness (avalia a qualidade da solução)
def fitness(threshold):
    preds = []

    for value in petal_length:
        if value < threshold:
            preds.append(0)  # setosa
        elif value < threshold + 2:
            preds.append(1)  # versicolor
        else:
            preds.append(2)  # virginica

    return accuracy_score(y, preds)

# criar população inicial
population = [random.uniform(1,6) for _ in range(10)]

# evoluir por algumas gerações
for generation in range(20):

    # avaliar população
    scores = [(ind, fitness(ind)) for ind in population]
    scores.sort(key=lambda x: x[1], reverse=True)

    # selecionar os melhores
    population = [ind for ind, score in scores[:5]]

    # cruzamento + mutação
    while len(population) < 10:
        parent1, parent2 = random.sample(population[:5], 2)
        child = (parent1 + parent2) / 2

        # mutação
        child += random.uniform(-0.2, 0.2)

        population.append(child)

# melhor solução
best = max(population, key=fitness)

print("Melhor threshold encontrado:", best)
print("Acurácia:", fitness(best))

Melhor threshold encontrado: 2.8404387788624192
Acurácia: 0.9533333333333334
